# Aurora Supply: shared features

Question: can the demand model and replenishment workflow reuse the same historical feature contract? Open the native Feature Store to inspect `aurora_inventory`, then query two product SKUs here. The data is synthetic and dated December 31, 2025. No live inventory or current-date forecast is implied.


In [ ]:
import json, ssl, urllib.request, urllib.error
from pathlib import Path

base = "https://feast-aurora-features-online.ai-showroom.svc.cluster.local"
context = ssl.create_default_context(cafile="/etc/service-ca/service-ca.crt")
token = Path("/var/run/secrets/kubernetes.io/serviceaccount/token").read_text().strip()

def query_features(skus, authenticated=True):
    payload = {"features": ["aurora_inventory:stock", "aurora_inventory:units_last_7d", "aurora_inventory:forecast_7d_units"],
               "entities": {"sku": skus}}
    headers = {"Content-Type": "application/json"}
    if authenticated:
        headers["Authorization"] = "Bearer " + token
    request = urllib.request.Request(base + "/get-online-features", data=json.dumps(payload).encode(), headers=headers)
    with urllib.request.urlopen(request, context=context, timeout=20) as response:
        return json.load(response)


In [ ]:
result = query_features(["AS-001", "AS-002"])
names = result["metadata"]["feature_names"]
columns = [item["values"] for item in result["results"]]
rows = [dict(zip(names, values)) for values in zip(*columns)]
print(json.dumps(rows, indent=2))
assert rows[0]["stock"] == 45
assert rows[0]["units_last_7d"] == 129
assert rows[0]["forecast_7d_units"] == 130.26


## Change the product

Replace a SKU above with another product from `data/products.json`. Compare its feature values with the catalog and measured Ray forecast. Explain why a shared definition helps the two teams, while forecast accuracy still needs a separate quality check.


In [ ]:
try:
    query_features(["AS-001"], authenticated=False)
    raise AssertionError("Anonymous feature access unexpectedly succeeded")
except urllib.error.HTTPError as error:
    assert error.code == 401, error.code
    print("PASS: anonymous feature access is rejected (401).")
